# 🎯 [MOV02_M07] 위험 경로 강행 또는 우회 결정을 위한 수송통제관 HITL 승인 워크플로우 설계
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M07 |
| **미션 주제** | 위험 경로 강행 또는 우회 결정을 위한 수송통제관 HITL 승인 워크플로우 설계 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 위험 경로 강행 또는 우회 결정을 위한 수송통제관 HITL 승인 워크플로우 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

우회 경로 승인을 위한 수송통제관 HITL 워크플로우를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **위험 경로 강행 또는 우회를 수송통제관이 직접 승인·반려하는 인간 개입(HITL) 결재 워크플로우를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AI가 제안한 경로는 모두 검토대기로 등록하고, 위협 구역을 지나는 강행 경로는 수송통제관과 상급 통제관 2단계 결재를 거치게 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/workflow/mov02_approval_system.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 보안 규정 및 행정 절차 부합성

#### 미션 PASS 조건

- 정량 기준: 이력 기록 무결성 100%, 승인 권한 우회 시도 차단율 100%


- 진행 기준: 결재선 승인/반려 시뮬레이션 및 이력 조회 화면 구현


- 검증 방법: 권한별 접근 제어(RBAC) 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""위험 경로 강행·우회 결정을 수송통제관이 직접 승인·반려하는 인간 개입(HITL) 결재 워크플로우를 구현한다. 역할별 권한(RBAC)과 결재선, 반려 사유 필수·재기안 연계, 해시 체인 이력을 두고 권한 우회 시도 10종 차단과 이력 무결성을 검증한다."""
import hashlib
import heapq
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    # [TODO : 학생 실습 7] 위험 경로 강행 또는 우회 결정을 위한 수송통제관 HITL 승인 워크플로우 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def edge_minutes(e, asset="truck"):
    # [TODO : 학생 실습 7] 위험 경로 강행 또는 우회 결정을 위한 수송통제관 HITL 승인 워크플로우 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network()
threats = make_threats()
DEPOT = "N0000"
for e in edges:
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
    e["risk"] = edge_risk(e, nodes, threats)


def propose(dest, lam):
    """AI 경로 제안: lam=0이면 최단시간(위협 통과 가능), lam>0이면 위협 회피 우회."""
    adj = build_adj(nodes, edges, lambda e: edge_minutes(e) + lam * 100 * e["risk"] if passable(e, 15) else None)
    _, path, es = dijkstra(adj, DEPOT, dest)
    surv = float(np.prod([1 - e["risk"] for e in es]))
    return {"path": path, "min": round(sum(edge_minutes(e) for e in es), 1), "surv": round(surv, 3), "threat": sum(e["risk"] > 0 for e in es)}


# ---------------- 사용자·역할(RBAC) ----------------
USERS = {"kim_req": "REQ", "lee_req": "REQ", "park_ctl": "CTRL", "choi_ctl": "CTRL", "jung_sup": "SUP", "han_aud": "AUD"}
PERM = {"REQ": {"submit", "redraft"}, "CTRL": {"approve", "reject", "submit"}, "SUP": {"approve", "reject"}, "AUD": {"read"}}


class Workflow:
    def __init__(self):
        self.docs, self.log, self.blocked = {}, [], []

    # 감사 로그: 앞 기록의 해시를 포함해 체인 형성(중간 수정 시 이후 해시가 모두 어긋남)
    def _audit(self, user, action, doc_id, detail):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        rec = {"seq": len(self.log) + 1, "user": user, "role": USERS.get(user, "?"), "action": action, "doc": doc_id, "detail": detail, "prev": prev}
        rec["hash"] = hashlib.sha256(json.dumps(rec, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16]
        self.log.append(rec)

    def verify(self):
        prev = "GENESIS"
        for r in self.log:
            body = {k: v for k, v in r.items() if k != "hash"}
            if r["prev"] != prev or hashlib.sha256(json.dumps(body, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16] != r["hash"]:
                return False, r["seq"]
            prev = r["hash"]
        return True, None

    def _deny(self, user, action, doc_id, why):
        self.blocked.append((user, action, doc_id, why))
        self._audit(user, "DENIED:" + action, doc_id, why)
        return False

    def submit(self, user, dest, route, parent=None):
        if "submit" not in PERM.get(USERS.get(user), set()) and not (parent and "redraft" in PERM.get(USERS.get(user), set())):
            return self._deny(user, "submit", "-", "권한 없음")
        doc_id = f"D{len(self.docs) + 1:03d}"
        risky = route["threat"] > 0 or route["surv"] < 0.9
        line = ["CTRL", "SUP"] if risky else ["CTRL"]          # 위험 경로 강행은 2단계 결재
        self.docs[doc_id] = {"drafter": user, "dest": dest, "route": route, "risky": risky, "line": line, "step": 0,
                             "status": "검토대기", "parent": parent, "approvers": []}
        self._audit(user, "submit", doc_id, f"{dest} 위협통과 {route['threat']} 생존 {route['surv']}")
        return doc_id

    def decide(self, user, doc_id, act, reason=""):
        role, d = USERS.get(user), self.docs.get(doc_id)
        if role is None:
            return self._deny(user, act, doc_id, "등록되지 않은 사용자")
        if d is None or d["status"] not in ("검토대기",):
            return self._deny(user, act, doc_id, "결재 대상 상태가 아님(이미 처리됨)")
        if act not in PERM[role]:
            return self._deny(user, act, doc_id, f"{role} 역할에 {act} 권한 없음")
        if d["line"][d["step"]] != role:
            return self._deny(user, act, doc_id, f"결재 순서 위반(현재 {d['line'][d['step']]} 차례)")
        if user == d["drafter"] or user in d["approvers"]:
            return self._deny(user, act, doc_id, "기안자·기결재자 중복 결재 금지")
        if len(reason.strip()) < 10:
            return self._deny(user, act, doc_id, "사유 10자 이상 필수")
        if act == "reject":
            d["status"] = "반려"
        else:
            d["approvers"].append(user); d["step"] += 1
            d["status"] = "승인" if d["step"] == len(d["line"]) else "검토대기"
        self._audit(user, act, doc_id, reason)
        return True

    def modify(self, user, doc_id, new_route):
        d = self.docs[doc_id]
        if d["status"] == "승인":
            return self._deny(user, "modify", doc_id, "승인 후 경로 변경 금지(재기안 필요)")
        d["route"] = new_route
        self._audit(user, "modify", doc_id, "경로 수정")
        return True


wf = Workflow()
# ---------------- 1) 정상 결재 시나리오 ----------------
cases = [("N0307", 0.0), ("N0307", 2.0), ("N0207", 0.0), ("N0507", 0.0), ("N0107", 2.0), ("N0407", 0.0)]
ids = [wf.submit("kim_req" if i % 2 == 0 else "lee_req", dst, propose(dst, lam)) for i, (dst, lam) in enumerate(cases)]
auto = [i for i in ids if wf.docs[i]["status"] == "승인"]
print(f"[1] AI 제안 {len(ids)}건 모두 '검토대기'로 등록, 자동 승인 {len(auto)}건 (목표 0건)")

wf.decide("park_ctl", ids[0], "reject", "위협 구역 통과 구간 3곳, 우회 경로로 재기안 바람")
new = wf.submit("kim_req", "N0307", propose("N0307", 2.0), parent=ids[0])               # 재기안(반려 문서와 연결)
wf.decide("park_ctl", new, "approve", "우회 경로 위협 통과 0, 기한 내 도착 가능")
wf.decide("park_ctl", ids[1], "approve", "위협 회피 우회 경로로 기한 충족")
wf.decide("choi_ctl", ids[2], "approve", "보급 긴급, 강행 필요성 인정 후 상급 결재 상신")
wf.decide("jung_sup", ids[2], "approve", "호송 간격 확대·통과 시간 단축 조건부 강행 승인")
wf.decide("park_ctl", ids[3], "reject", "생존확률 낮음, 다음 주기 우회로 재계획")
wf.decide("choi_ctl", ids[4], "approve", "안전 경로, 기한 충족 확인함")

# ---------------- 2) 권한 우회 시도 10종 ----------------
attacks = [("kim_req", ids[5], "approve", "급하니 제가 승인합니다", "요청자 자기 승인"),
           ("han_aud", ids[5], "approve", "감사 계정으로 승인 시도", "감사 역할 승인"),
           ("ghost", ids[5], "approve", "외부 계정 승인 시도입니다", "미등록 사용자"),
           ("park_ctl", ids[5], "approve", "", "사유 없는 승인"),
           ("park_ctl", ids[5], "reject", "x", "사유 없는 반려"),
           ("jung_sup", ids[5], "approve", "상급자가 먼저 바로 승인함", "결재 순서 건너뛰기"),
           ("park_ctl", ids[1], "approve", "같은 문서를 다시 승인한다", "처리 완료 문서 재승인")]
for u, doc, act, rs, name in attacks:
    wf.decide(u, doc, act, rs)
wf.decide("jung_sup", ids[2], "reject", "승인 뒤 반려로 뒤집기 시도")                          # 8 승인 완료 문서 번복
wf.modify("kim_req", ids[2], propose("N0207", 3.0))                                          # 9 승인 후 경로 바꿔치기
d6 = wf.submit("park_ctl", "N0607", propose("N0507", 0.0))
wf.decide("park_ctl", d6, "approve", "내가 기안한 문서를 직접 승인")                          # 10 기안자 자기 결재
n_attack = 10
print(f"[2] 권한 우회 시도 {n_attack}건 중 차단 {len(wf.blocked)}건 (차단율 {len(wf.blocked) / n_attack:.0%}, 목표 100%)")
for u, a, doc, why in wf.blocked:
    print(f"    차단 {u:<9} {a:<7} {doc}: {why}")

# ---------------- 3) 이력 조회·무결성 ----------------
ok, _ = wf.verify()
print(f"[3] 감사 로그 {len(wf.log)}건, 해시 체인 무결성 {'100%' if ok else '훼손'}")
hist = pd.DataFrame([{"seq": r["seq"], "사용자": r["user"], "행위": r["action"], "사유": r["detail"][:24]} for r in wf.log if r["doc"] in (ids[0], new)])
print(f"    {ids[0]} 반려 -> {new} 재기안 이력 조회:")
print(hist.to_string(index=False))
tam = json.loads(json.dumps(wf.log))
wf.log[4]["detail"] = "사유를 사후에 고쳐 씀"
ok2, at = wf.verify()
print(f"    5번 기록을 사후 수정 -> 무결성 검사 {'통과' if ok2 else f'실패(위변조 검출, {at}번 기록)'}")
wf.log = tam
st = pd.Series({k: v["status"] for k, v in wf.docs.items()})
print("[4] 문서 상태:", st.value_counts().to_dict(), "| 강행 승인 문서는 2단계(통제관+상급 통제관) 결재 완료분만 집행")
print("결재는 판단 보조 기록이며, 실제 수송 강행 여부는 지휘계통의 결심에 따른다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
st.value_counts().plot.bar(ax=ax[0], color="tab:blue"); ax[0].set_title("Document status (no auto-approval)"); ax[0].tick_params(axis="x", rotation=0)
ax[0].set_xticklabels(["pending" if t.get_text() == "검토대기" else "approved" if t.get_text() == "승인" else "rejected" for t in ax[0].get_xticklabels()])
reasons = pd.Series([w for *_, w in wf.blocked]).str.slice(0, 6).value_counts()
ax[1].barh(range(len(reasons)), reasons.values, color="tab:red"); ax[1].set_yticks(range(len(reasons))); ax[1].set_yticklabels([f"rule {i + 1}" for i in range(len(reasons))])
ax[1].set_title("Blocked bypass attempts by rule"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[1] AI 제안 6건 모두 '검토대기'로 등록, 자동 승인 0건 (목표 0건)
[2] 권한 우회 시도 10건 중 차단 10건 (차단율 100%, 목표 100%)
    차단 kim_req   approve D006: REQ 역할에 approve 권한 없음
    차단 han_aud   approve D006: AUD 역할에 approve 권한 없음
    차단 ghost     approve D006: 등록되지 않은 사용자
    차단 park_ctl  approve D006: 사유 10자 이상 필수
    차단 park_ctl  reject  D006: 사유 10자 이상 필수
    차단 jung_sup  approve D006: 결재 순서 위반(현재 CTRL 차례)
    차단 park_ctl  approve D002: 결재 대상 상태가 아님(이미 처리됨)
    차단 jung_sup  reject  D003: 결재 대상 상태가 아님(이미 처리됨)
    차단 kim_req   modify  D003: 승인 후 경로 변경 금지(재기안 필요)
    차단 park_ctl  approve D008: 기안자·기결재자 중복 결재 금지
[3] 감사 로그 25건, 해시 체인 무결성 100%
    D001 반려 -> D007 재기안 이력 조회:
 seq      사용자      행위                       사유
   1  kim_req  submit    N0307 위협통과 3 생존 0.578
   7 park_ctl  reject 위협 구역 통과 구간 3곳, 우회 경로로 재
   8  kim_req  submit      N0307 위협통과 0 생존 1.0
   9 park_ctl approve 우회 경로 위협 통과 0, 기한 내 도착 가
    5번 기록을 사후 수정 -> 무결성 검사 실패(위변조 검출, 5번 기록)
[4] 문서 상태: {'승인': 4, '반려': 2, '검토대기': 2} | 강행 승인 문서는 2단계(통제관+상급 통제관) 결재 완료분만 집행
결재는 판단 보조 기록이며, 실제 수송 강행 여부는 지휘계통의 결심에 따른다.
```
</details>
